# PMLDL Assignment 2 (2026): Qwen3 Dense Decoder-Only Transformer

## Task description

Implement a small **decoder-only language model with the Qwen3 Dense architecture** in PyTorch and train it **from random initialization** to translate English into Portuguese. You will implement the architectural components yourself, then use the training and evaluation scaffold below.

The architecture reference for this assignment is **Qwen3 Dense**, specifically the full-attention design used by Qwen3-0.6B. “Dense” means that every token passes through the same feed-forward network in each layer. There are no experts or routing. Qwen2/Qwen2.5 and later Qwen variants are not interchangeable architecture specifications.

This is a scaled-down educational model: use a smaller vocabulary, hidden size, layer count, and context window. Reproducing Qwen's pretrained weights, large-scale training, chat template, or thinking behavior is not required. The task remains English → Portuguese translation.

## Architecture contract and references

Read the architecture section of the [Qwen3 Technical Report](https://arxiv.org/abs/2505.09388), the [official Qwen3 release](https://qwenlm.github.io/blog/qwen3/), and the [Qwen3-0.6B configuration](https://huggingface.co/Qwen/Qwen3-0.6B/blob/main/config.json). The [Transformers v4.51.3 Qwen3 implementation](https://github.com/huggingface/transformers/blob/v4.51.3/src/transformers/models/qwen3/modeling_qwen3.py) is a reference for conventions; write your own implementation and cite any resources you consult.

Your model must contain:

- One shared token vocabulary and embedding, followed by a stack of causal decoder blocks.
- Grouped-query attention (**GQA**) with fewer key/value heads than query heads.
- Per-head **QK-Norm** before rotary position embeddings (**RoPE**), applied to queries and keys only.
- **RMSNorm** before each attention/MLP sublayer, residual connections, and a final RMSNorm.
- A dense **SwiGLU** MLP. Attention and MLP projections have no bias.
- A vocabulary projection (LM head) producing next-token logits. Tie its weight to the token embedding, as in Qwen3-0.6B.

Use full causal attention in every block, with zero dropout for the baseline. Do not add an encoder, cross-attention, additive sinusoidal embeddings, embedding scaling by $\sqrt{d_{model}}$, or learned absolute position embeddings. KV caching is an optional optimization; correct generation by recomputing the prefix is sufficient.

## Allowed building blocks

You may use PyTorch tensor operations, autograd, `nn.Module`, `nn.ModuleList`, `nn.Parameter`, `nn.Linear`, `nn.Embedding`, `nn.SiLU`/`F.silu`, loss functions, and optimizers. Implement RMSNorm, RoPE, head grouping, attention scores, causal/padding masks, and the decoder blocks yourself.

Do not use `nn.Transformer*`, `nn.MultiheadAttention`, `F.scaled_dot_product_attention`, `nn.RMSNorm`, ready-made Qwen/Llama/other Transformer layers, or a pretrained language model. Do not copy a model implementation. Tokenization and dataset libraries are allowed.

## Submission

Provide your solution in **this notebook and submit it to Moodle**. Replace every `NotImplementedError` and complete the experiment/report cells. Run the completed notebook from a fresh kernel, save its outputs, and include:

- Your model implementation.
- The seed, package versions, device, data split sizes, tokenizer vocabulary size, model configuration, parameter count, training budget, and training/validation loss curves.
- Test loss, test perplexity, and **autoregressive corpus BLEU**; at least five test translations with English inputs and Portuguese references.
- A short discussion of differences from the original encoder–decoder Transformer, training behavior, errors, and limitations. State any departures from the scaffold.

Save the tokenizer and best validation checkpoint, and include code to reload them. If Moodle has a size limit, provide an accessible link to these artifacts in the notebook. Choose epochs and hyperparameters using validation data; evaluate the test set only after selecting the model.

## Grading criteria — 5 points

| Criterion | Points | Evidence |
| --- | ---: | --- |
| RMSNorm, RoPE, and dense SwiGLU | 1.0 | Correct formulas, tensor shapes, and bias settings |
| Causal GQA with QK-Norm | 1.0 | Separate Q/K/V projections, correct KV grouping, QK-Norm order, and causal/padding masks |
| Complete Qwen3 Dense causal LM | 1.0 | Pre-norm residual blocks, independent layer parameters, final RMSNorm, and tied LM head |
| Decoder-only translation and training | 1.0 | Correct sequence construction, target-only shifted loss, successful training, validation history, and checkpoint |
| Autoregressive evaluation and analysis | 1.0 | Test loss/perplexity, corpus BLEU, five translations, and a comparison of Qwen3 Dense with the classic Transformer |

After the submission deadline, an **in-person defence with TAs** will be arranged. You must demonstrate your model's English → Portuguese translation, explain your implementation and training/evaluation code, and answer questions about your solution.

## 1. Environment

The notebook contains intentional implementation gaps. Data preparation can run independently; training becomes runnable after you implement the required classes. A GPU is recommended for the experiment; use a smaller configuration for CPU debugging.

In [ ]:
%pip install torch datasets tokenizers sacrebleu tqdm matplotlib


In [ ]:
import importlib.metadata
import math
import random
from dataclasses import asdict, dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import sacrebleu
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)
print({name: importlib.metadata.version(name)
       for name in ["torch", "datasets", "tokenizers", "sacrebleu"]})


## 2. A small Qwen3 Dense configuration

All tensors use batch-first layout. Let $B$ be batch size, $T$ sequence length, $D$ hidden size, $H_q$ query head count, $H_{kv}$ key/value head count, $d_h$ head dimension, and $I$ MLP intermediate size. Require $H_q$ to be divisible by $H_{kv}$ and $d_h$ to be even.

Keep `head_dim` explicit: Qwen3 permits $H_q d_h \ne D$. Q maps $D \to H_q d_h$, K/V map $D \to H_{kv}d_h$, and the attention output projection maps $H_q d_h \to D$. The baseline below uses $H_q d_h=D$ for a small experiment. You may reduce width/depth for debugging, but keep GQA with $H_{kv}<H_q$ in the submitted model.

In [ ]:
@dataclass
class QwenDenseConfig:
    vocab_size: int
    hidden_size: int = 256
    intermediate_size: int = 768
    num_hidden_layers: int = 4
    num_attention_heads: int = 8
    num_key_value_heads: int = 2
    head_dim: int = 32
    max_position_embeddings: int = 256
    rms_norm_eps: float = 1e-6
    rope_theta: float = 1_000_000.0
    tie_word_embeddings: bool = True

    def __post_init__(self):
        assert self.num_attention_heads % self.num_key_value_heads == 0
        assert self.num_key_value_heads < self.num_attention_heads
        assert self.head_dim % 2 == 0
        assert self.vocab_size > 0


## 3. Implement the components

### RMSNorm

For a vector $x$ of width $d$, use
$$\operatorname{RMSNorm}(x)=\gamma\odot\frac{x}{\sqrt{\frac{1}{d}\sum_{i=1}^{d}x_i^2+\epsilon}}.$$
Here $\gamma$ is a learned vector initialized to ones, $\epsilon$ prevents division by zero, and $\odot$ is elementwise multiplication. Compute the mean square in float32 for numerical stability, then restore the input dtype. Do not subtract the mean or add a bias. Hidden-state norms have width $D$; Q/K norms have width $d_h$ and share their learned scale across heads.

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()

        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        input_dtype = x.dtype
        variance = x.float().pow(2).mean(dim=-1, keepdim=True)
        x = x.float() * torch.rsqrt(variance + self.eps)
        return self.weight * x.to(input_dtype)

### Rotary position embeddings (RoPE)

Use the Qwen split-half convention. For position $p$ and $i=0,\ldots,d_h/2-1$, define $\theta_i=\text{rope\_theta}^{-2i/d_h}$ and angle $p\theta_i$. Duplicate the half-width angles to width $d_h$. With $\operatorname{rotate\_half}([a,b])=[-b,a]$ for equal-width halves,
$$\operatorname{RoPE}(x,p)=x\odot\cos(p\theta)+\operatorname{rotate\_half}(x)\odot\sin(p\theta).$$

Compute frequencies/angles in float32 on the input device, then cast cosine/sine to the Q/K dtype. Apply RoPE after QK-Norm, never to V. Use absolute positions starting at zero; generation must use the same position convention as training.

In [ ]:
class RotaryEmbedding(nn.Module):
    def __init__(self, head_dim, theta=1_000_000.0):
        super().__init__()

        if head_dim % 2 != 0:
            raise ValueError("head_dim must be even")

        inv_freq = 1.0 / (
            theta ** (torch.arange(0, head_dim, 2).float() / head_dim)
        )

        self.register_buffer("inv_freq", inv_freq, persistent=False)

    @staticmethod
    def rotate_half(x):
        x1, x2 = x.chunk(2, dim=-1)
        return torch.cat((-x2, x1), dim=-1)

    def forward(self, q, k, position_ids):
        freqs = position_ids.to(self.inv_freq.dtype).unsqueeze(-1) * self.inv_freq
        angles = torch.cat((freqs, freqs), dim=-1)

        cos = angles.cos().unsqueeze(1)
        sin = angles.sin().unsqueeze(1)

        q_rotated = q * cos.to(q.dtype) + self.rotate_half(q) * sin.to(q.dtype)
        k_rotated = k * cos.to(k.dtype) + self.rotate_half(k) * sin.to(k.dtype)

        return q_rotated, k_rotated

### Grouped-query causal attention and QK-Norm

Project hidden states to Q, K, V and reshape to $[B,H_q,T,d_h]$, $[B,H_{kv},T,d_h]$, $[B,H_{kv},T,d_h]$. Apply separate learned per-head RMSNorm modules to Q and K, then apply RoPE. Each KV head serves a consecutive group of $H_q/H_{kv}$ query heads; logically repeat K/V heads to match Q.

For query position $t$ and key position $s$, allow attention only when $s\le t$ **and** `attention_mask[b, s] == 1`. Here the mask is $[B,T]$, with 1 for real tokens and 0 for right padding. Apply this rule in every forward pass, including validation and generation. Compute
$$A=\operatorname{softmax}(QK^\top/\sqrt{d_h}+M),\qquad O=AV,$$
where $M$ is zero for allowed entries and $-\infty$ otherwise. Compute softmax in float32, cast back, concatenate query heads, and project to $D$. Right-padded queries can attend to preceding real tokens; their outputs are excluded from the loss. Every example below has a nonempty prompt, so no query needs an entirely masked row.

In [ ]:
class GroupedQueryAttention(nn.Module):
    def __init__(self, config):
        super().__init__()

        self.num_heads = config.num_attention_heads
        self.num_kv_heads = config.num_key_value_heads
        self.head_dim = config.head_dim
        self.num_groups = self.num_heads // self.num_kv_heads

        self.q_proj = nn.Linear(
            config.hidden_size,
            self.num_heads * self.head_dim,
            bias=False
        )

        self.k_proj = nn.Linear(
            config.hidden_size,
            self.num_kv_heads * self.head_dim,
            bias=False
        )

        self.v_proj = nn.Linear(
            config.hidden_size,
            self.num_kv_heads * self.head_dim,
            bias=False
        )

        self.o_proj = nn.Linear(
            self.num_heads * self.head_dim,
            config.hidden_size,
            bias=False
        )

        self.q_norm = RMSNorm(self.head_dim, config.rms_norm_eps)
        self.k_norm = RMSNorm(self.head_dim, config.rms_norm_eps)

        self.rope = RotaryEmbedding(self.head_dim, config.rope_theta)

    def forward(self, hidden_states, attention_mask, position_ids):
        batch_size, seq_len, _ = hidden_states.shape

        q = self.q_proj(hidden_states)
        k = self.k_proj(hidden_states)
        v = self.v_proj(hidden_states)

        q = q.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view(batch_size, seq_len, self.num_kv_heads, self.head_dim).transpose(1, 2)
        v = v.view(batch_size, seq_len, self.num_kv_heads, self.head_dim).transpose(1, 2)

        q = self.q_norm(q)
        k = self.k_norm(k)

        q, k = self.rope(q, k, position_ids)

        k = k.repeat_interleave(self.num_groups, dim=1)
        v = v.repeat_interleave(self.num_groups, dim=1)

        scores = torch.matmul(q, k.transpose(-2, -1))
        scores = scores / math.sqrt(self.head_dim)

        causal_mask = torch.ones(
            seq_len, seq_len,
            device=hidden_states.device,
            dtype=torch.bool
        ).tril()

        valid_keys = attention_mask[:, None, None, :].bool()
        allowed = causal_mask[None, None, :, :] & valid_keys

        scores = scores.float().masked_fill(~allowed, float("-inf"))

        weights = torch.softmax(scores, dim=-1).to(v.dtype)

        output = torch.matmul(weights, v)

        output = output.transpose(1, 2).contiguous()
        output = output.view(batch_size, seq_len, self.num_heads * self.head_dim)

        return self.o_proj(output)

### Dense SwiGLU and pre-norm decoder block

The MLP is $\operatorname{SwiGLU}(x)=W_{down}(\operatorname{SiLU}(W_{gate}x)\odot W_{up}x)$, with two projections $D\to I$ and one $I\to D$, all without bias. Every token uses this same MLP.

A decoder block computes
$$u=x+\operatorname{Attention}(\operatorname{RMSNorm}_1(x)),\qquad
y=u+\operatorname{SwiGLU}(\operatorname{RMSNorm}_2(u)).$$
The two sublayer norms have independent learned scales. Different decoder layers must also have independent parameters.

In [ ]:
class SwiGLU(nn.Module):
    def __init__(self, config):
        super().__init__()

        self.gate_proj = nn.Linear(
            config.hidden_size,
            config.intermediate_size,
            bias=False
        )

        self.up_proj = nn.Linear(
            config.hidden_size,
            config.intermediate_size,
            bias=False
        )

        self.down_proj = nn.Linear(
            config.intermediate_size,
            config.hidden_size,
            bias=False
        )

    def forward(self, x):
        gate = F.silu(self.gate_proj(x))
        up = self.up_proj(x)

        return self.down_proj(gate * up)


class QwenDenseDecoderLayer(nn.Module):
    def __init__(self, config):
        super().__init__()

        self.input_layernorm = RMSNorm(
            config.hidden_size,
            config.rms_norm_eps
        )

        self.self_attn = GroupedQueryAttention(config)

        self.post_attention_layernorm = RMSNorm(
            config.hidden_size,
            config.rms_norm_eps
        )

        self.mlp = SwiGLU(config)

    def forward(self, x, attention_mask, position_ids):
        residual = x

        x = self.input_layernorm(x)
        x = self.self_attn(x, attention_mask, position_ids)
        x = residual + x

        residual = x

        x = self.post_attention_layernorm(x)
        x = self.mlp(x)
        x = residual + x

        return x

### Complete causal language model

Embed `input_ids`, pass through the decoder stack, apply final RMSNorm and the tied bias-free LM head. Return raw logits `[B, T, vocab_size]`; do not apply softmax before cross-entropy. Create positions `0, ..., T-1` for each right-padded example. Reject sequences longer than the configured context window.

Initialize linear/embedding weights from a normal distribution with standard deviation 0.02 and RMSNorm scales to one. Tie embedding/head weights after initialization. The scaffold applies the one-token shift externally, so `forward` must not shift tokens or labels.

In [ ]:
class QwenDenseForCausalLM(nn.Module):
    def __init__(self, config):
        super().__init__()

        self.config = config

        self.embed_tokens = nn.Embedding(
            config.vocab_size,
            config.hidden_size
        )

        self.layers = nn.ModuleList([
            QwenDenseDecoderLayer(config)
            for _ in range(config.num_hidden_layers)
        ])

        self.norm = RMSNorm(
            config.hidden_size,
            config.rms_norm_eps
        )

        self.lm_head = nn.Linear(
            config.hidden_size,
            config.vocab_size,
            bias=False
        )

        self.apply(self._init_weights)

        if config.tie_word_embeddings:
            self.lm_head.weight = self.embed_tokens.weight

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

        elif isinstance(module, RMSNorm):
            nn.init.ones_(module.weight)

    def forward(self, input_ids, attention_mask=None):
        batch_size, seq_len = input_ids.shape

        if seq_len > self.config.max_position_embeddings:
            raise ValueError("Sequence exceeds maximum context length")

        if attention_mask is None:
            attention_mask = torch.ones_like(input_ids)

        position_ids = torch.arange(
            seq_len,
            device=input_ids.device
        ).unsqueeze(0).expand(batch_size, -1)

        x = self.embed_tokens(input_ids)

        for layer in self.layers:
            x = layer(x, attention_mask, position_ids)

        x = self.norm(x)

        logits = self.lm_head(x)

        return logits

In [ ]:
assert torch.cuda.is_available(), "CUDA is not available"

test_config = QwenDenseConfig(vocab_size=8000)
test_model = QwenDenseForCausalLM(test_config).to("cuda")

input_ids = torch.randint(0, 8000, (2, 10), device="cuda")
attention_mask = torch.ones_like(input_ids)
attention_mask[1, 7:] = 0

logits = test_model(input_ids, attention_mask)

assert logits.shape == (2, 10, 8000)
assert torch.isfinite(logits).all()
assert test_model.lm_head.weight is test_model.embed_tokens.weight

assert (
    test_model.layers[0].self_attn.q_proj.weight
    is not test_model.layers[1].self_attn.q_proj.weight
)

changed_ids = input_ids.clone()
changed_ids[:, 5:] = torch.randint(0, 8000, (2, 5), device="cuda")

with torch.no_grad():
    original = test_model(input_ids, attention_mask)
    changed = test_model(changed_ids, attention_mask)

torch.testing.assert_close(
    original[:, :5],
    changed[:, :5],
    rtol=1e-5,
    atol=1e-5
)

loss = F.cross_entropy(
    logits[:, :-1].reshape(-1, 8000),
    input_ids[:, 1:].reshape(-1)
)

loss.backward()

assert torch.isfinite(loss)
assert test_model.embed_tokens.weight.grad is not None
assert test_model.layers[0].self_attn.q_proj.weight.grad is not None
assert test_model.layers[0].mlp.gate_proj.weight.grad is not None

print("GPU:", torch.cuda.get_device_name(0))
print("Parameters:", sum(p.numel() for p in test_model.parameters()))
print("Output shape:", logits.shape)
print("Loss:", loss.item())
print("Architecture: OK")
print("Causal attention: OK")
print("Weight tying: OK")
print("Gradients: OK")

del test_model, logits, original, changed, loss
torch.cuda.empty_cache()

## 4. Translation as causal language modeling

Use one sequence per pair:

`<bos> <en> English tokens <pt> Portuguese tokens <eos>`

The prompt ends at `<pt>`. During teacher forcing, Portuguese tokens are available only to later positions through causal attention. Optimize next-token cross-entropy **only on Portuguese tokens and EOS**. The English prompt still participates in attention.

For full sequence `s` and prompt length `P`, construct full-length labels as `[-100] * P + s[P:]`. Feed `s[:-1]` to the model and compare its logits to `labels[1:]`. Thus the logit at `<pt>` predicts the first Portuguese token. `-100` means ignored by the loss; it is never an input token ID. Do not shift a second time inside the model.

Use [OPUS-100 en-pt](https://huggingface.co/datasets/Helsinki-NLP/opus-100) with its official train, validation, and test splits. The scaffold samples up to 50k/1k/1k pairs with a fixed seed and excludes normalized holdout sentences from training. The repository contains a much larger training split, so the budget can be increased for additional experiments.

In [ ]:
from datasets import load_dataset
import unicodedata

# Keep the corpus-provided split boundaries. Never split the combined corpus.
corpus = load_dataset("Helsinki-NLP/opus-100", "en-pt")

MAX_TRAIN_PAIRS = 50_000
MAX_VALIDATION_PAIRS = 1_000
MAX_TEST_PAIRS = 1_000

def normalized_text(text):
    return " ".join(unicodedata.normalize("NFKC", text).casefold().split())

def clean_pairs(rows, limit, excluded_en=None, excluded_pt=None):
    excluded_en = excluded_en or set()
    excluded_pt = excluded_pt or set()
    
    seen, result = set(), []
    
    for row in rows:
        en, pt = (row["translation"][lang].strip() for lang in ("en", "pt"))
        en_key, pt_key = normalized_text(en), normalized_text(pt)
        key = (en_key, pt_key)
        
        if not en_key or not pt_key or key in seen:
            continue
        
        if en_key in excluded_en or pt_key in excluded_pt:
            continue
        
        seen.add(key)
        result.append({"translation": {"en": en, "pt": pt}})
        
        if len(result) == limit:
            break
        
    return result

# Exclude all official holdout sentences, even those outside the evaluation budget.
holdout_en, holdout_pt = set(), set()

for split in ("validation", "test"):
    for row in corpus[split]:
        holdout_en.add(normalized_text(row["translation"]["en"]))
        holdout_pt.add(normalized_text(row["translation"]["pt"]))
        
train_pairs = clean_pairs(corpus["train"].shuffle(seed=SEED), MAX_TRAIN_PAIRS,
                          holdout_en, holdout_pt)
val_pairs = clean_pairs(corpus["validation"].shuffle(seed=SEED), MAX_VALIDATION_PAIRS)

val_en = {normalized_text(row["translation"]["en"]) for row in corpus["validation"]}
val_pt = {normalized_text(row["translation"]["pt"]) for row in corpus["validation"]}

test_pairs = clean_pairs(corpus["test"].shuffle(seed=SEED), MAX_TEST_PAIRS, val_en, val_pt)

assert len(train_pairs) == MAX_TRAIN_PAIRS

print({"train": len(train_pairs), "validation": len(val_pairs), "test": len(test_pairs)})
print("Source splits:", {split: len(rows) for split, rows in corpus.items()})


### A shared byte-level BPE tokenizer

To keep embedding/logit memory small, train an 8k tokenizer **using the training split only**, with both languages sharing one vocabulary. This replaces Qwen's production tokenizer; it does not change the decoder architecture. Special-token IDs are local to this tokenizer. Save it with the checkpoint.

The scaffold tokenizes each language separately and inserts boundary IDs explicitly. Use exactly the same prompt construction during training and generation.

In [ ]:
from tokenizers import Tokenizer, models, pre_tokenizers, decoders, trainers

tokenizer = Tokenizer(models.BPE(unk_token="<unk>"))

tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
tokenizer.decoder = decoders.ByteLevel()

special_tokens = ["<pad>", "<unk>", "<bos>", "<eos>", "<en>", "<pt>"]

trainer = trainers.BpeTrainer(
    vocab_size=8_000, min_frequency=2, special_tokens=special_tokens,
    initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),
)

def training_texts():
    for pair in train_pairs:
        yield pair["translation"]["en"]
        yield pair["translation"]["pt"]

tokenizer.train_from_iterator(training_texts(), trainer=trainer)
pad_id, unk_id, bos_id, eos_id, en_id, pt_id = [
    tokenizer.token_to_id(t) for t in special_tokens
]

ARTIFACT_DIR = Path("assignment2_opus100_artifacts")

ARTIFACT_DIR.mkdir(exist_ok=True)
tokenizer.save(str(ARTIFACT_DIR / "tokenizer.json"))

print("Vocabulary:", tokenizer.get_vocab_size())


### Length budget and batches

Use at most 96 English and 96 Portuguese tokens. Filter out longer or empty tokenized pairs rather than training on incomplete translations. The full sequence has at most 196 tokens and fits the 256-token context window. Report retained counts. These limits define the evaluation subset too; do not describe its score as a full-corpus score. Preserve the official corpus split boundaries. The scaffold excludes normalized exact holdout sentences from training; near-duplicates or noisy alignments can still remain.

In [ ]:
MAX_SOURCE_TOKENS = 96
MAX_TARGET_TOKENS = 96
MAX_NEW_TOKENS = MAX_TARGET_TOKENS + 1  # Allow EOS after the longest target.

def make_prompt(english):
    return [bos_id, en_id] + tokenizer.encode(english).ids + [pt_id]

def encode_pairs(pairs):
    examples = []
    
    for pair in pairs:
        english, portuguese = (pair["translation"][lang] for lang in ("en", "pt"))
        source_ids = tokenizer.encode(english).ids
        target_ids = tokenizer.encode(portuguese).ids
        
        if not (0 < len(source_ids) <= MAX_SOURCE_TOKENS
                and 0 < len(target_ids) <= MAX_TARGET_TOKENS):
            continue
        
        prompt = make_prompt(english)
        sequence = prompt + target_ids + [eos_id]
        
        examples.append({
            "input_ids": sequence,
            "labels": [-100] * len(prompt) + target_ids + [eos_id],
            "prompt_ids": prompt,
            "english": english, "portuguese": portuguese,
        })
        
    return examples

train_examples = encode_pairs(train_pairs)
val_examples = encode_pairs(val_pairs)
test_examples = encode_pairs(test_pairs)

print({"retained_train": len(train_examples), "retained_validation": len(val_examples),
       "retained_test": len(test_examples)})

assert all([train_examples, val_examples, test_examples]), "A split became empty"

def collate_fn(examples):
    length = max(len(ex["input_ids"]) for ex in examples)
    ids = torch.full((len(examples), length), pad_id, dtype=torch.long)
    labels = torch.full_like(ids, -100)
    mask = torch.zeros_like(ids)
    
    for row, ex in enumerate(examples):
        n = len(ex["input_ids"])
        
        ids[row, :n] = torch.tensor(ex["input_ids"])
        labels[row, :n] = torch.tensor(ex["labels"])
        mask[row, :n] = 1
        
    return {"input_ids": ids[:, :-1], "attention_mask": mask[:, :-1],
            "labels": labels[:, 1:]}

BATCH_SIZE = 32

train_loader = DataLoader(train_examples, batch_size=BATCH_SIZE, shuffle=True,
                          collate_fn=collate_fn,
                          generator=torch.Generator().manual_seed(SEED))
val_loader = DataLoader(val_examples, batch_size=BATCH_SIZE, collate_fn=collate_fn)
test_loader = DataLoader(test_examples, batch_size=BATCH_SIZE, collate_fn=collate_fn)

example = train_examples[0]

print("Prompt:", tokenizer.decode(example["prompt_ids"], skip_special_tokens=False))
print("Supervised continuation:", tokenizer.decode(
    [t for t in example["labels"] if t != -100], skip_special_tokens=False))
print({k: tuple(v.shape) for k, v in next(iter(train_loader)).items()})


## 5. Training and validation

Create a fresh model for this experiment. The scaffold averages loss over supervised tokens, clips gradients, and selects the checkpoint by validation loss. Set a realistic training budget, record it, and plot the history. One epoch is only a debugging run; report actual convergence behavior.

The baseline uses up to 50k training pairs and a maximum of 10 epochs. Stop after two epochs without a validation-loss improvement of at least 0.001. Save the best validation checkpoint even when training stops early.

In [ ]:
config = QwenDenseConfig(vocab_size=tokenizer.get_vocab_size())

assert MAX_SOURCE_TOKENS + MAX_TARGET_TOKENS + 4 <= config.max_position_embeddings

model = QwenDenseForCausalLM(config).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.01)

print(asdict(config))
print("Parameters:", sum(p.numel() for p in model.parameters()))

def batch_loss(model, batch):
    batch = {k: v.to(device) for k, v in batch.items()}
    
    logits = model(batch["input_ids"], attention_mask=batch["attention_mask"])
    labels = batch["labels"]
    
    loss_sum = F.cross_entropy(logits.reshape(-1, logits.size(-1)),
                               labels.reshape(-1), ignore_index=-100, reduction="sum")
    count = (labels != -100).sum().item()
    
    assert count > 0
    
    return loss_sum, count

@torch.no_grad()
def evaluate_loss(model, loader):
    model.eval()
    
    total_loss, total_tokens = 0.0, 0
    
    for batch in loader:
        loss_sum, count = batch_loss(model, batch)
        
        total_loss += loss_sum.item()
        total_tokens += count
        
    return total_loss / total_tokens

def train_epoch(model, loader, optimizer):
    model.train()
    
    total_loss, total_tokens = 0.0, 0
    
    for batch in tqdm(loader):
        optimizer.zero_grad(set_to_none=True)
        
        loss_sum, count = batch_loss(model, batch)
        
        loss = loss_sum / count
        
        if not torch.isfinite(loss):
            raise RuntimeError("Non-finite training loss")
        
        loss.backward()
        
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        
        optimizer.step()
        
        total_loss += loss_sum.item()
        total_tokens += count
        
    return total_loss / total_tokens


In [ ]:
NUM_EPOCHS = 10  # Adjust using validation evidence and your compute budget.
history = []
best_val = float("inf")
PATIENCE = 2
MIN_DELTA = 1e-3
stale_epochs = 0

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss = train_epoch(model, train_loader, optimizer)
    val_loss = evaluate_loss(model, val_loader)
    previous_best = best_val
    
    history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss})
    
    print(history[-1])
    
    if val_loss < best_val:
        best_val = val_loss
        
        torch.save({"model": model.state_dict(), "config": asdict(config),
                    "epoch": epoch, "val_loss": val_loss, "seed": SEED},
                   ARTIFACT_DIR / "best.pt")

    if val_loss < previous_best - MIN_DELTA:
        stale_epochs = 0
    else:
        stale_epochs += 1

    if stale_epochs >= PATIENCE:
        print("Early stopping: validation loss stopped improving")
        break

plt.plot([r["epoch"] for r in history], [r["train_loss"] for r in history], label="train")
plt.plot([r["epoch"] for r in history], [r["val_loss"] for r in history], label="validation")
plt.xlabel("Epoch")
plt.ylabel("Target-token cross-entropy")
plt.legend()
plt.show()


## 6. Autoregressive generation and held-out evaluation

At inference time the model receives **only the English prompt ending in `<pt>`**. Generate one token at a time, append it to the prefix, and stop at EOS or the declared token budget. Never pass a reference translation to this function. Keep the causal mask active. The baseline uses greedy decoding and recomputes the prefix.

Teacher-forced argmax predictions are not generated translations and must not be used for BLEU. For corpus BLEU, pass one list of hypotheses and a list containing the complete aligned reference stream: `sacrebleu.corpus_bleu(hypotheses, [references])`.

**Test loss** is the average next-token cross-entropy over Portuguese target tokens and EOS in the test set. English prompt tokens and padding are excluded. Test perplexity is `exp(test_loss)`.

In [ ]:
@torch.no_grad()
def translate_sentence(model, english, max_new_tokens=MAX_NEW_TOKENS):
    model.eval()
    
    prompt = make_prompt(english)
    
    if len(prompt) + max_new_tokens > config.max_position_embeddings:
        raise ValueError("Prompt and generation budget exceed the context window")
    
    ids = torch.tensor([prompt], dtype=torch.long, device=device)
    generated = []
    
    for _ in range(max_new_tokens):
        logits = model(ids, attention_mask=torch.ones_like(ids))
        
        next_id = int(logits[0, -1].argmax().item())
        
        if next_id == eos_id:
            break
        
        generated.append(next_id)
        
        ids = torch.cat([ids, ids.new_tensor([[next_id]])], dim=1)
        
    return tokenizer.decode(generated, skip_special_tokens=True)

# Reload the saved tokenizer and selected checkpoint before test evaluation.
tokenizer = Tokenizer.from_file(str(ARTIFACT_DIR / "tokenizer.json"))

checkpoint = torch.load(ARTIFACT_DIR / "best.pt", map_location=device, weights_only=True)
config = QwenDenseConfig(**checkpoint["config"])

model = QwenDenseForCausalLM(config).to(device)
model.load_state_dict(checkpoint["model"])

print("Selected epoch:", checkpoint["epoch"])

test_loss = evaluate_loss(model, test_loader)

print("Test target-token loss:", test_loss)
print("Test target-token perplexity:", math.exp(test_loss))

hypotheses = [translate_sentence(model, ex["english"]) for ex in tqdm(test_examples)]
references = [ex["portuguese"] for ex in test_examples]

bleu_metric = sacrebleu.metrics.BLEU()
bleu = bleu_metric.corpus_score(hypotheses, [references])

print("Autoregressive test corpus BLEU:", bleu.score)
print("BLEU signature:", bleu_metric.get_signature())

for ex, hypothesis in list(zip(test_examples, hypotheses))[:5]:
    print("English:", ex["english"])
    print("Generated Portuguese:", hypothesis)
    print("Reference Portuguese:", ex["portuguese"])
    print()


## 7. Report and discussion

Explain how the Qwen3 Dense decoder-only architecture differs from the classic encoder–decoder Transformer.

**Your analysis:**

TODO